# 01 Factibilidad

Que fraccion de los valores del model-index aparece en el cuerpo del README.

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
HF_TOKEN = os.environ["HF_TOKEN"]

In [3]:
import os, re
import pandas as pd, requests
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.environ["HF_TOKEN"]
CSV = "../data/HFCO2_2026.csv"

# 1) Carga: prueba ambos separadores y verifica que exista la columna
for sep in (";", ","):
    df = pd.read_csv(CSV, sep=sep, encoding="utf-8-sig")
    if "performance_metrics" in df.columns:
        break
else:
    raise KeyError(f"No encuentro 'performance_metrics'. Columnas: {list(df.columns)}")
print(f"separador={sep!r}  filas={len(df)}")

# 2) Modelos con performance_metrics vacío
EMPTY = {"", "[]", "{}", "null", "none", "nan"}
def is_empty(pm):
    return pd.isna(pm) or str(pm).strip().lower() in EMPTY

empty = df[df["performance_metrics"].map(is_empty)]
print(f"{len(empty)} de {len(df)} con performance_metrics vacío")
sample = empty.sample(min(20, len(empty)), random_state=42)

# 3) Descarga y limpieza del README
def readme(model_id):
    r = requests.get(f"https://huggingface.co/{model_id}/raw/main/README.md",
                     headers={"Authorization": f"Bearer {HF_TOKEN}"}, timeout=30)
    return r.text if r.ok else None

def body(text):  # quita el YAML del inicio
    return re.sub(r"\A---\n.*?\n---\n", "", text, count=1, flags=re.S)

# 4) Evidencia de métricas en el cuerpo
NAMES = ["accuracy", "acc", r"f1(?:[- ]score)?", "f-score", "precision", "recall",
         r"(?:roc[- ])?auc", "bleu", r"rouge[-_ ]?[12l]?", "meteor", "wer", "cer",
         "perplexity", "ppl", r"exact[- ]match", "mrr", "ndcg", "spearman",
         "pearson", "mse", "rmse", "mae", "r2"]
STRONG = "(?:" + "|".join(NAMES) + ")"
SEP = r"[\s:=|*_]*(?:of|is|was)?[\s:=|*_]*"
INLINE = re.compile(rf"\b({STRONG}|loss)\b{SEP}(\d+(?:\.\d+)?)", re.I)
HEADER = re.compile(rf"\b{STRONG}\b", re.I)

def inline_hits(text):
    return [(m.group(1).lower(), float(m.group(2))) for m in INLINE.finditer(text)]

def table_rows(text):  # filas numéricas bajo una tabla con encabezado de métrica
    lines, n = text.splitlines(), 0
    for i in range(len(lines) - 2):
        if (lines[i].count("|") >= 2 and HEADER.search(lines[i])
                and re.match(r"\s*\|?\s*:?-{2,}", lines[i + 1])):
            j = i + 2
            while j < len(lines) and lines[j].count("|") >= 2:
                n += bool(re.search(r"\d", lines[j]))
                j += 1
    return n

rows = []
for _, r in sample.iterrows():
    t = readme(r["modelId"])
    if t is None:                       # sin README: no cuenta como 0
        rows.append((r["modelId"], None, None, None, None, "")); continue
    b = body(t)
    hits = inline_hits(b)
    strong = [h for h in hits if h[0] != "loss"]
    n_tab = table_rows(b)
    rows.append((r["modelId"], len(strong), n_tab, len(hits) - len(strong),
                 len(strong) > 0 or n_tab > 0,
                 "; ".join(f"{k}={v}" for k, v in strong[:3])))

res = pd.DataFrame(rows, columns=["modelId", "n_inline", "n_filas_tabla", "n_loss",
                                  "con_evidencia", "muestra"])
pd.set_option("display.width", 200, "display.max_colwidth", 60)
print(res.sort_values("n_inline"))
ok = res["n_inline"].notna()
print(f"\nREADME no descargado: {(~ok).sum()} de {len(res)}")
print(f"Con evidencia de métricas: {res['con_evidencia'].eq(True).sum()} de {ok.sum()}")

separador=','  filas=2057
1533 de 2057 con performance_metrics vacío
                                                        modelId  n_inline  n_filas_tabla  n_loss con_evidencia                                        muestra
4                                DarwinAnim8or/GPT-NoSleep-355m       0.0            0.0     0.0         False                                               
8                     Manolo/autotrain-nl-en-esco-1-78131140807       0.0            0.0     1.0         False                                               
17                BrianDsouzaAI/autotrain-tab-multi-92337144714       0.0            0.0     1.0         False                                               
13                     sindri101/radiology-report-en-zh-ft-base       0.0            0.0     1.0         False                                               
16                            Tritkoman/EnglishtoAncientGreekV4       0.0            0.0     1.0         False                               

In [4]:
import os, pandas as pd
from dotenv import load_dotenv
from metric_extraction.download import download_readmes

load_dotenv()
df = pd.read_csv("../data/HFCO2_2026.csv", sep=",")
status = download_readmes(df["modelId"].tolist(), "../data/raw/readmes",
                          token=os.environ["HF_TOKEN"])
print(status["result"].value_counts())
print(status["http"].value_counts())

100/2057
200/2057
300/2057
400/2057
500/2057
600/2057
700/2057
800/2057
900/2057
1000/2057
1100/2057
1200/2057
1300/2057
1400/2057
1500/2057
1600/2057
1700/2057
1800/2057
1900/2057
2000/2057
2057/2057
result
ok      1931
fail     126
Name: count, dtype: int64
http
200    1931
403      94
404      32
Name: count, dtype: int64


In [5]:
from pathlib import Path
from metric_extraction.evidence import read_body, signature, table_rows
from metric_extraction.download import safe_name

rows = []
for _, r in df.iterrows():
    p = Path("../data/raw/readmes") / f"{safe_name(r['modelId'])}.md"
    if not p.exists():
        continue
    b = read_body(p)
    rows.append((r["modelId"], r["pipeline_tag"], signature(b), table_rows(b)))

sig = pd.DataFrame(rows, columns=["modelId", "pipeline_tag", "firma", "filas_tabla"])
sig["tiene_firma"] = sig["firma"].map(len) > 0
print(f"{sig['tiene_firma'].mean():.1%} de {len(sig)} fichas con alguna métrica")

top = sig.loc[sig["tiene_firma"], "firma"].value_counts()
print(f"{len(top)} firmas distintas")
print((top.cumsum() / top.sum()).head(10).round(2))   # cobertura acumulada
print(sig.groupby("pipeline_tag")["tiene_firma"].agg(["mean", "size"])
         .sort_values("size", ascending=False).head(12))

NameError: name 'df' is not defined

In [4]:
# 1) ¿De qué familias es la referencia?
ref = df[df["performance_metrics"].notna()
         & (df["performance_metrics"].astype(str).str.strip().str.len() > 2)]
print(ref["pipeline_tag"].value_counts().head(8))
print(ref["auto"].value_counts())

# 2) ¿Qué hay en los README de las tareas con 0 %?
import re
zero = sig[sig["pipeline_tag"].isin(["sentence-similarity", "feature-extraction",
                                     "translation", "text-generation"])
           & ~sig["tiene_firma"]].groupby("pipeline_tag").head(2)
for _, r in zero.iterrows():
    b = read_body(Path("../data/raw/readmes") / f"{safe_name(r['modelId'])}.md")
    lines = [l for l in b.splitlines() if re.search(r"\d\.\d{2,}", l)][:6]
    print("=====", r["pipeline_tag"], r["modelId"])
    print("\n".join(l[:140] for l in lines))

NameError: name 'df' is not defined

In [3]:
from pathlib import Path
from metric_extraction.evidence import read_body
from metric_extraction.download import safe_name
from metric_extraction.data import ref_values, value_in_text

ref = df[df["performance_metrics"].notna()].copy()
ref["vals"] = ref["performance_metrics"].map(ref_values)
ref = ref[ref["vals"].notna()]
print(len(ref), "fichas con valores de referencia")

out = []
for _, r in ref.iterrows():
    p = Path("../data/raw/readmes") / f"{safe_name(r['modelId'])}.md"
    if not p.exists():
        continue
    b = read_body(p)
    found = sum(value_in_text(v, b) for v in r["vals"])
    out.append((r["modelId"], r["pipeline_tag"], r["auto"], len(r["vals"]), found / len(r["vals"])))

cov = pd.DataFrame(out, columns=["modelId", "pipeline_tag", "auto", "n_vals", "frac"])
print(cov["frac"].describe().round(2))
print(f"{(cov['frac'] >= 0.5).mean():.1%} de las fichas con >=50% de sus valores en el cuerpo")
print(cov.groupby("pipeline_tag")["frac"].agg(["median", "size"])
         .sort_values("size", ascending=False).head(8).round(2))

NameError: name 'df' is not defined

In [2]:
import random
rng = random.Random(42)
ids = cov["modelId"].tolist()
null = []
for _, r in ref[ref["modelId"].isin(ids)].iterrows():
    other = rng.choice([i for i in ids if i != r["modelId"]])
    b = read_body(Path("../data/raw/readmes") / f"{safe_name(other)}.md")
    null.append(sum(value_in_text(v, b) for v in r["vals"]) / len(r["vals"]))
print(pd.Series(null).describe().round(2))

NameError: name 'cov' is not defined

In [1]:
from collections import Counter
from metric_extraction.evidence import read_body, inline_hits
from metric_extraction.gold import gold_tuples, score_hits, value_close

rows, miss = [], Counter()
for _, r in ref.iterrows():
    p = Path("../data/raw/readmes") / f"{safe_name(r['modelId'])}.md"
    if not p.exists():
        continue
    gold = gold_tuples(r["performance_metrics"])
    if not gold:
        continue
    hits = inline_hits(read_body(p))
    rec, n_gold, ok, n_hits = score_hits(hits, gold)
    rows.append((r["modelId"], r["pipeline_tag"], rec, n_gold, ok, n_hits))
    for al, rv in gold:                       # métricas de referencia no recuperadas
        if not any(n in al and value_close(v, rv) for n, v in hits):
            miss[sorted(al)[0]] += 1

sc = pd.DataFrame(rows, columns=["modelId", "pipeline_tag", "rec", "n_gold", "ok", "n_hits"])
tot = sc[["rec", "n_gold", "ok", "n_hits"]].sum()
P, R = tot["ok"] / tot["n_hits"], tot["rec"] / tot["n_gold"]
print(f"{len(sc)} fichas | P={P:.3f}  R={R:.3f}  F1={2*P*R/(P+R):.3f}")

g = sc.groupby("pipeline_tag").agg(cards=("modelId", "size"), rec=("rec", "sum"),
        n_gold=("n_gold", "sum"), ok=("ok", "sum"), n_hits=("n_hits", "sum"))
g["P"] = g["ok"] / g["n_hits"]; g["R"] = g["rec"] / g["n_gold"]
g["F1"] = 2 * g["P"] * g["R"] / (g["P"] + g["R"])
print(g[["cards", "P", "R", "F1"]].sort_values("cards", ascending=False).head(8).round(2))
print(miss.most_common(20))

NameError: name 'ref' is not defined